In [15]:
%pip install xgboost


In [16]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(r"../data/subjects.csv")
print(df.shape)

In [17]:
# 피처 선택

# 1단계 : y2(호감 강도)와 상관 0.5 이상 남기기

corr = df.select_dtypes(include='number').corr()
target_corr = corr['y2_attraction_intensity'].drop(['y1_attracted', 'y2_attraction_intensity', 'y3_dated', 'y4_relationship_satisfaction']).abs()

strong = target_corr[target_corr >= 0.5]

print(f'1단계 : {len(target_corr)}개 -> {len(strong)}개 (y2와 관련 있는 것만)')


In [18]:
# 2단계 : 피처끼리 상관 0.8 이상이면 하나 제거로 다중공선성 제거
strong_corr = df[strong.index].corr()
mask = np.triu(np.ones(strong_corr.shape), k=1).astype(bool)
pairs = strong_corr.where(mask).stack()
high_pairs = pairs[pairs >= 0.8]


to_drop = set()

for (a, b), val in high_pairs.items():
    if abs(target_corr[a]) >= abs(target_corr[b]):
        to_drop.add(b)
    else:
        to_drop.add(a)

selected = [c for c in strong.index if c not in to_drop]

print(f'2단계 : {len(strong)}개 => {len(selected)}개 (중복제거)')

In [ ]:
# 3단계: RF importance로 실제 예측 기여도 상위 5개

X = df[selected]
y = df['y2_attraction_intensity']

rf = RandomForestRegressor(n_estimators=100, random_state=33)
rf.fit(X,y)

importance = pd.DataFrame({
    'feature':selected,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)


top5 = importance.head(5)['feature'].tolist()

print(f'3단계 : {len(selected)}개 -> {len(top5)}개 (예측 기여도 상위)')
print(f'\n 최종 피쳐: {top5}')








In [21]:
# 셀 3
X_final = df[top5]
y = df['y2_attraction_intensity']

model = XGBRegressor(n_estimators=100, random_state=42)
model.fit(X_final, y)

In [22]:
#몬테카를로 시뮬레이션
# "이상형(y2 >=8) 만날 확률은?"


# 5개 피처의 분포 추출
means = df[top5].mean()
stds = df[top5].std()


# 가상 인물 1000명 생성

np.random.seed(42)
n_sim = 1000
simulated = pd.DataFrame(
    np.random.normal(loc=means, scale=stds, size=(n_sim, len(top5))),
    columns=top5
)

simulated = pd.DataFrame(
    np.random.normal(loc=means, scale=stds, size=(n_sim, len(top5))),
    columns=top5
)


pred = model.predict(simulated)

threshold = 8 
hit_rate = (pred >=threshold).mean() * 100
print( f'이상형(y2 >= {threshold}) 확률 : {hit_rate:.1f}%')
print(f'= 약 {int(1000/max(hit_rate*10, 1))}명 만나면 1명')

In [23]:
for t in [10, 9, 8, 7, 6, 5]:
    rate = (pred >= t).mean() * 100
    print(f'y2 >= {t}: {rate:5.1f}%')

In [24]:
baseline_rate = (pred >= threshold).mean()*100


sensitivity = {}
for feat in top5:
    # 기본 시뮬레이션 데이터 복사

    sim_adjusted = simulated.copy()
    sim_adjusted[feat] = sim_adjusted[feat] + 1
    sim_adjusted = sim_adjusted.clip(1, 10)

    pred_adjusted = model.predict(sim_adjusted)
    new_rate = (pred_adjusted >= threshold).mean()*100

    sensitivity[feat] = {
        'baseline': baseline_rate,
        'adjusted': new_rate,
        'delta': new_rate - baseline_rate
    }

sens_df = pd.DataFrame(sensitivity).T.sort_values('delta', ascending=False)


print('=== 피처별 민감도 (1점 타협 시 이상형 확률 변화) ===')
print(sens_df.round(2))
print(f'\n가장 가성비 좋은 타협: {sens_df.index[0]}')

In [25]:
# 셀 7: 진짜 타협 시뮬레이터 — 조합 + 비선형

# (1) 2개 피처 동시 타협 효과
from itertools import combinations

combo_results = {}
for f1, f2 in combinations(top5, 2):
    sim_adj = simulated.copy()
    sim_adj[f1] = sim_adj[f1] + 1
    sim_adj[f2] = sim_adj[f2] + 1
    sim_adj = sim_adj.clip(1, 10)
    rate = (model.predict(sim_adj) >= threshold).mean() * 100
    combo_results[f'{f1} + {f2}'] = rate

combo_df = pd.Series(combo_results).sort_values(ascending=False)
print(f'=== 기본 확률: {baseline_rate:.1f}% ===\n')
print('--- 2피처 동시 타협 ---')
for combo, rate in combo_df.items():
    print(f'{combo}: {rate:.1f}% (+{rate - baseline_rate:.1f}%p)')

In [26]:
# 셀 8: 시너지 분석
synergy = {}
for combo_name, combo_rate in combo_results.items():
    f1, f2 = combo_name.split(' + ')
    individual_sum = sensitivity[f1]['delta'] + sensitivity[f2]['delta']
    actual = combo_rate - baseline_rate
    synergy[combo_name] = {
        'expected': round(individual_sum, 1),
        'actual': round(actual, 1),
        'synergy': round(actual - individual_sum, 1)
    }

syn_df = pd.DataFrame(synergy).T.sort_values('synergy', ascending=False)
print('=== 시너지 분석 (실제 - 예상) ===')
print(syn_df)

In [27]:
# 셀 9: 타협 시뮬레이터 시각화

# 한글 폰트 설정
import matplotlib.pyplot as plt
import platform
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 피처 한글명 매핑
feat_kr = {
    'E34_push_pull_tension': '밀당 텐션',
    'G45_conversation_fun': '대화 재미',
    'B9_scent_sensory': '향/감각',
    'D21_humor_match': '유머 코드',
    'D24_clumsy_but_trying': '서툰 노력'
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- 왼쪽: 단독 타협 효과 ---
ax1 = axes[0]
labels = [feat_kr[f] for f in sens_df.index]
colors = ['#e74c3c' if d > 3 else '#3498db' if d > 1.5 else '#95a5a6' 
          for d in sens_df['delta']]
bars = ax1.barh(labels, sens_df['delta'], color=colors)
for bar, val in zip(bars, sens_df['delta']):
    ax1.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
             f'+{val:.1f}%p', va='center', fontsize=11)
ax1.set_xlabel('확률 변화 (%p)')
ax1.set_title(f'단독 1점 타협 효과 (기본: {baseline_rate:.1f}%)')

# --- 오른쪽: 시너지 TOP 5 ---
ax2 = axes[1]
top_syn = syn_df.head(5)
combo_kr = []
for name in top_syn.index:
    f1, f2 = name.split(' + ')
    combo_kr.append(f'{feat_kr[f1]} + {feat_kr[f2]}')

x = range(len(top_syn))
ax2.bar(x, top_syn['expected'], width=0.35, label='예상 (단순합산)', color='#bdc3c7', align='center')
ax2.bar([i+0.35 for i in x], top_syn['actual'], width=0.35, label='실제', color='#2ecc71', align='center')
ax2.set_xticks([i+0.175 for i in x])
ax2.set_xticklabels(combo_kr, rotation=25, ha='right', fontsize=9)
ax2.set_ylabel('확률 변화 (%p)')
ax2.set_title('2피처 동시 타협: 예상 vs 실제 (시너지)')
ax2.legend()

# 시너지 값 표시
for i, syn_val in enumerate(top_syn['synergy']):
    if syn_val > 0:
        ax2.text(i+0.35, top_syn['actual'].iloc[i] + 0.2, 
                f'+{syn_val}', ha='center', fontsize=10, color='#e74c3c', fontweight='bold')

plt.suptitle('타협 시뮬레이터', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('compromise_simulator.png', dpi=150, bbox_inches='tight')
plt.show()

In [28]:
# 셀 10: 타협 시뮬레이터 — 확률 언어 시각화

fig, ax = plt.subplots(figsize=(12, 7))

# 시나리오별 확률 계산
scenarios = {}

# 0. 기본
scenarios['타협 없음'] = baseline_rate

# 1. 단독 타협 (상위 3개만)
for feat in sens_df.index[:3]:
    sim_adj = simulated.copy()
    sim_adj[feat] = sim_adj[feat] + 1
    sim_adj = sim_adj.clip(1, 10)
    rate = (model.predict(sim_adj) >= threshold).mean() * 100
    scenarios[f'{feat_kr[feat]} 1점 양보'] = rate

# 2. 베스트 조합
sim_adj = simulated.copy()
sim_adj['E34_push_pull_tension'] = sim_adj['E34_push_pull_tension'] + 1
sim_adj['D21_humor_match'] = sim_adj['D21_humor_match'] + 1
sim_adj = sim_adj.clip(1, 10)
scenarios['밀당 + 유머 동시 양보'] = (model.predict(sim_adj) >= threshold).mean() * 100

# 3. 올인 (전부 1점씩)
sim_adj = simulated.copy()
for feat in top5:
    sim_adj[feat] = sim_adj[feat] + 1
sim_adj = sim_adj.clip(1, 10)
scenarios['5개 전부 1점 양보'] = (model.predict(sim_adj) >= threshold).mean() * 100

# 정렬
labels = list(scenarios.keys())
rates = list(scenarios.values())
people = [f'{int(round(100/r))}명 중 1명' if r > 0 else '∞' for r in rates]

# 그래프
colors = ['#95a5a6'] + ['#3498db']*3 + ['#2ecc71'] + ['#e74c3c']
bars = ax.barh(range(len(labels)), rates, color=colors[:len(labels)])

# 확률 + "N명 중 1명" 동시 표시
for i, (bar, rate, ppl) in enumerate(zip(bars, rates, people)):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{rate:.1f}%  →  {ppl}', va='center', fontsize=12, fontweight='bold')

ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=11)
ax.set_xlabel('이상형 확률 (%)')
ax.set_title('타협 시뮬레이터: 뭘 양보하면 얼마나 달라지나?', fontsize=14, fontweight='bold')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('compromise_simulator_v2.png', dpi=150, bbox_inches='tight')
plt.show()